# Краткое описание
- Цель эксперимента - первичный EDA и проверка качества спанов токсичных комментариев.
- Данные - multilingual-toxic-spans (данные с HF, только русский язык).
- Основные выводы - распределения количества токсичных слов и длины текстов.

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

sns.set(style="whitegrid")

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("textdetox/multilingual_toxic_spans", split="ru")

In [ ]:
df = pd.DataFrame(ds)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.describe(include="object")

In [ ]:
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Duplicate sentences:", df["Sentence"].duplicated().sum())
print("Duplicate negative connotations:", df["Negative Connotations"].duplicated().sum())

In [ ]:
print("Duplicate pairs:", df.duplicated(subset=["Sentence", "Negative Connotations"]).sum())

Проверка на аномалии

In [ ]:

import re

def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def count_tokens(text):
    text = safe_str(text)
    if not text:
        return 0
    return len([t for t in re.split(r"\s+", text) if t])

def count_connotations(text):
    text = safe_str(text)
    if not text:
        return 0
    return len([x.strip() for x in text.split(",") if x.strip()])

def anomaly_report(df):
    report = {}

    for col in ["Sentence", "Negative Connotations"]:
        report[f"{col}_missing"] = int(df[col].isna().sum())
        report[f"{col}_empty"] = int(df[col].astype(str).str.strip().eq("").sum())
        report[f"{col}_non_string"] = int((~df[col].apply(lambda x: isinstance(x, str)) & df[col].notna()).sum())

    report["duplicate_rows"] = int(df.duplicated().sum())
    report["duplicate_sentence"] = int(df["Sentence"].duplicated().sum())
    report["duplicate_negative_connotations"] = int(df["Negative Connotations"].duplicated().sum())
    report["duplicate_pairs"] = int(df.duplicated(subset=["Sentence", "Negative Connotations"]).sum())

    df_anom = df.copy()

    df_anom["sentence_len_words"] = df_anom["Sentence"].apply(count_tokens)
    df_anom["sentence_len_chars"] = df_anom["Sentence"].apply(lambda x: len(safe_str(x)))
    df_anom["neg_count"] = df_anom["Negative Connotations"].apply(count_connotations)
    df_anom["neg_len_words"] = df_anom["Negative Connotations"].apply(count_tokens)
    df_anom["neg_len_chars"] = df_anom["Negative Connotations"].apply(lambda x: len(safe_str(x)))

    report["sentence_len_words_zero"] = int((df_anom["sentence_len_words"] == 0).sum())
    report["neg_count_zero"] = int((df_anom["neg_count"] == 0).sum())
    report["neg_count_gt_10"] = int((df_anom["neg_count"] > 10).sum())
    report["sentence_len_words_gt_50"] = int((df_anom["sentence_len_words"] > 50).sum())
    report["sentence_len_chars_gt_300"] = int((df_anom["sentence_len_chars"] > 300).sum())

    weird_mask = (
        df_anom["Sentence"].str.contains(r"#ERROR!|nan|None|<NA>", case=False, na=False) |
        df_anom["Negative Connotations"].str.contains(r"#ERROR!|nan|None|<NA>", case=False, na=False)
    )
    report["weird_text_patterns"] = int(weird_mask.sum())

    print("Anomaly report:")
    for k, v in report.items():
        print(f"{k}: {v}")

    print("\nExamples of suspicious rows:")
    display(
        df_anom.loc[
            (df_anom["sentence_len_words"] == 0) |
            (df_anom["neg_count"] == 0) |
            weird_mask |
            (df_anom["neg_count"] > 10) |
            (df_anom["sentence_len_words"] > 50)
        , ["Sentence", "Negative Connotations", "sentence_len_words", "neg_count"]].head(15)
    )

    return df_anom, report

df_checked, anomaly_stats = anomaly_report(df)

Анализ длины предложений и списка токсичных слов

In [ ]:
df["sentence_len_chars"] = df["Sentence"].astype(str).str.len()
df["sentence_len_words"] = df["Sentence"].astype(str).str.split().apply(len)

df["neg_len_chars"] = df["Negative Connotations"].astype(str).str.len()
df["neg_len_words"] = df["Negative Connotations"].astype(str).str.split().apply(len)

In [ ]:
df[["sentence_len_chars", "sentence_len_words", "neg_len_chars", "neg_len_words"]].describe()

Визуализация длины текста

In [ ]:
from pathlib import Path

def _find_project_dir(name="project", max_levels=10):
    p = Path.cwd()
    for _ in range(max_levels):
        if p.name == name:
            return p
        p = p.parent
    for ancestor in Path.cwd().parents:
        if ancestor.name == name:
            return ancestor
    return Path.cwd()

project_dir = _find_project_dir()
output_dir = project_dir / "artifacts" / "EDA" / "multilabel_dataset"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df["sentence_len_words"], bins=40, kde=True, color="steelblue")
plt.title("Distribution of sentence length in words")
plt.xlabel("Words")
plt.ylabel("Count")
plt.savefig(str(output_dir / "sentence_length_words_distribution.png"))
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df["neg_len_words"], bins=40, kde=True, color="tomato")
plt.title("Distribution of negative connotations length in words")
plt.xlabel("Words")
plt.ylabel("Count")
plt.savefig(str(output_dir / "negative_connotations_length_words_distribution.png"))
plt.show()

In [ ]:
length_df = pd.DataFrame({
    "type": ["Sentence"] * len(df) + ["Negative Connotations"] * len(df),
    "length_words": list(df["sentence_len_words"]) + list(df["neg_len_words"])
})

plt.figure(figsize=(9, 5))
sns.boxplot(data=length_df, x="type", y="length_words")
plt.title("Length comparison")
plt.xlabel("")
plt.ylabel("Words")
plt.xticks(rotation=15)
plt.savefig(str(output_dir / "sentence_negative_length_comparison_boxplot.png"))
plt.show()

Анализ количества токсичных слов в одном примере

In [ ]:
df["neg_count"] = df["Negative Connotations"].astype(str).apply(lambda x: len([i.strip() for i in x.split(",") if i.strip()]))
df["neg_count"].describe()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["neg_count"], bins=20, kde=True, color="darkred")
plt.title("Distribution of number of negative connotations per row")
plt.xlabel("Count")
plt.ylabel("Rows")
plt.savefig(str(output_dir / "negative_connotations_count_distribution.png"))
plt.show()

Самые частые негативные слова

In [ ]:
neg_words = []

for text in df["Negative Connotations"].astype(str):
    parts = [x.strip().lower() for x in text.split(",")]
    neg_words.extend([x for x in parts if x])

neg_counter = Counter(neg_words)

neg_top = pd.DataFrame(neg_counter.most_common(20), columns=["word", "count"])
neg_top

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=neg_top, y="word", x="count", color="darkred")
plt.title("Top 20 negative connotations")
plt.xlabel("Count")
plt.ylabel("")
plt.savefig(str(output_dir / "top_20_negative_connotations.png"))
plt.show()

Частотный анализ слов в самих предложениях

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^а-яёa-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
df["sentence_clean"] = df["Sentence"].apply(preprocess_text)

In [ ]:
sentence_words = " ".join(df["sentence_clean"]).split()
sentence_counter = Counter(sentence_words)

sentence_top = pd.DataFrame(sentence_counter.most_common(20), columns=["word", "count"])
sentence_top

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=sentence_top, y="word", x="count", color="steelblue")
plt.title("Top 20 words in sentences")
plt.xlabel("Count")
plt.ylabel("")
plt.savefig(str(output_dir / "top_20_sentence_words.png"))
plt.show()

Биграммы в предложениях

In [ ]:
def get_top_ngrams(text_series, n=2, top_k=20):
    vectorizer = CountVectorizer(ngram_range=(n, n))
    X = vectorizer.fit_transform(text_series)
    freqs = X.sum(axis=0).A1
    ngrams = vectorizer.get_feature_names_out()
    result = pd.DataFrame({"ngram": ngrams, "count": freqs})
    return result.sort_values("count", ascending=False).head(top_k)

In [ ]:
top_bigrams = get_top_ngrams(df["sentence_clean"], n=2, top_k=20)
top_bigrams

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_bigrams, y="ngram", x="count", color="steelblue")
plt.title("Top 20 bigrams in sentences")
plt.xlabel("Count")
plt.ylabel("")
plt.savefig(str(output_dir / "top_20_bigrams.png"))
plt.show()
